# Project Test Console — Google Colab

Ayarlar ve kullanım **Colab'ın kendi form alanlarıdır**; bu notebook'ta ipywidgets veya ayrı bir web paneli kullanılmaz. Başlangıç ayarları Tessera'nın portable test profiline göre hazırlanmıştır. Başka projelerde Repository / Ref ve test komutlarını değiştirmen yeterli.

## Kullanım
1. Bu notebook'u Google Colab'da aç.
2. Üstteki form alanlarından repository, branch, secret, zaman aşımı, kurulum ve test komutlarını düzenle.
3. Private repository için Colab → **Secrets** alanında `GH_TOKEN` oluştur ve notebook erişimini aç. Fine-grained token için yalnızca ilgili repo üzerinde `Contents: Read-only` yeterlidir. Public repo için token gerekmez.
4. Form ayarlarını kaydedip kod hücresini çalıştır. İleride ayarları değiştirdiğinde aynı hücreyi yeniden çalıştırabilirsin.
5. Sonuç raporu hücre çıktısında tek bir metin olarak oluşur. Colab çıktısındaki kopyalama seçeneğini kullan veya rapor metnini seçip kopyalayarak doğrudan ChatGPT'ye yapıştır.

## Komut biçimi
Her test alanı şu biçimde doldurulur:

```text
Test adı :: shell komutu
Clang testi :: CXX=clang++ ./test.sh :: needs=clang++
```

`needs=` bölümü isteğe bağlıdır. İstenen program bulunamazsa test `SKIP` olur. Clang kurulumu varsayılan olarak açıktır ve yalnızca Clang testi tanımlanmış, `clang++` eksikse çalışır. İstersen formdan kapatabilirsin.

## Rapor ve güvenlik
Rapor repository, ref, kesin commit SHA, ortam, süre, komut, exit kodları, stdout/stderr ve Issue Pool içerir. Rapor `/content/PROJECT_VALIDATION_REPORT.txt` konumuna da kaydedilir.

Token yalnızca Git klonlama sırasında geçici askpass ortamıyla kullanılır; klonlama bittiğinde ortamdan çıkarılır ve test süreçlerine aktarılmaz. Yalnızca güvendiğin repository ve komutları çalıştır: kurulum ve test komutları Colab çalışma alanında shell komutu olarak yürütülür.

## Tessera test kapsamı
Varsayılan testler GCC strict mock, Clang strict mock ve GCC ASan/UBSan'dır. Bunlar Linux üzerinde mock Win32 backend doğrulamasıdır; gerçek Windows/MSVC, Detours, inline-hook, x64 unwind, DLL unload veya loader davranışını kanıtlamaz. Bu native testler Windows ortamında ayrıca yürütülmelidir.

In [ ]:
#@title Project Test Console — Colab Native Settings { display-mode: "form" }
#@markdown **Ayarlar doğrudan Google Colab form alanlarından yapılır.** Varsayılan profil Tessera içindir; yeni projelerde repository, branch ve komutları değiştirmen yeterli.
#@markdown
#@markdown ### Repository ve erişim
repo_url = "https://github.com/Jue-dev/Tessera.git" #@param {type:"string"}
branch_or_ref = "issue-6/unwind-lifecycle" #@param {type:"string"}
secret_name = "GH_TOKEN" #@param {type:"string"}
git_username = "x-access-token" #@param {type:"string"}
clone_depth = 1 #@param {type:"integer"}
clone_timeout_seconds = 240 #@param {type:"integer"}
#@markdown ### Çalıştırma ayarları
test_timeout_seconds = 900 #@param {type:"integer"}
install_clang_if_missing = True #@param {type:"boolean"}
continue_after_failure = True #@param {type:"boolean"}
keep_workspace = True #@param {type:"boolean"}
#@markdown ### İsteğe bağlı kurulum komutları — boş bırakılabilir
setup_command_1 = "" #@param {type:"string"}
setup_command_2 = "" #@param {type:"string"}
setup_command_3 = "" #@param {type:"string"}
#@markdown ### Test komutları — her alan: Test adı :: komut :: needs=program (son bölüm isteğe bağlı)
test_1 = "GCC · strict mock :: bash verify/windows_mock/run_windows_mock_tests.sh" #@param {type:"string"}
test_2 = "Clang · strict mock :: CXX=clang++ bash verify/windows_mock/run_windows_mock_tests.sh :: needs=clang++" #@param {type:"string"}
test_3 = "GCC · ASan + UBSan :: EXTRA_CXXFLAGS='-fsanitize=address,undefined -fno-omit-frame-pointer' ASAN_OPTIONS='detect_leaks=1:halt_on_error=1' UBSAN_OPTIONS='halt_on_error=1:print_stacktrace=1' bash verify/windows_mock/run_windows_mock_tests.sh" #@param {type:"string"}
test_4 = "" #@param {type:"string"}
test_5 = "" #@param {type:"string"}

import os
import re
import json
import time
import shutil
import tempfile
import subprocess
import platform
import datetime
import pathlib
import urllib.parse
from google.colab import userdata

# The Colab form above is the configuration interface. This cell is the runner.
WORKDIR = pathlib.Path("/content/project-validation-workspace")
REPORT_PATH = pathlib.Path("/content/PROJECT_VALIDATION_REPORT.txt")
started_all = time.time()
issues, setup_results, test_results = [], [], []
commit = "NOT_AVAILABLE"
clone_ok = False
secret = None

def redact(value, token):
    value = str(value or "")
    if token:
        value = value.replace(token, "[REDACTED]")
    return value

def clip_output(value, limit=22000):
    value = str(value or "")
    if len(value) <= limit:
        return value
    half = (limit - 64) // 2
    return value[:half] + "\n\n[OUTPUT TRUNCATED]\n\n" + value[-half:]

def run_shell(command, cwd, env, timeout):
    start = time.time()
    try:
        result = subprocess.run(
            command,
            cwd=str(cwd),
            env=env,
            shell=True,
            executable="/bin/bash",
            capture_output=True,
            text=True,
            timeout=timeout
        )
        output = (result.stdout or "")
        if result.stderr:
            output += ("\n[STDERR]\n" if output else "") + result.stderr
        return result.returncode, round(time.time() - start, 2), output
    except subprocess.TimeoutExpired as exc:
        stdout = exc.stdout.decode("utf-8", "replace") if isinstance(exc.stdout, bytes) else (exc.stdout or "")
        stderr = exc.stderr.decode("utf-8", "replace") if isinstance(exc.stderr, bytes) else (exc.stderr or "")
        return 124, round(time.time() - start, 2), f"TIMEOUT after {timeout}s\n{stdout}\n[STDERR]\n{stderr}"
    except Exception as exc:
        return -1, round(time.time() - start, 2), f"{type(exc).__name__}: {exc}"

def parse_test(spec, index):
    spec = spec.strip()
    if not spec:
        return None, None
    parts = [part.strip() for part in spec.split("::", 2)]
    if len(parts) < 2 or not parts[0] or not parts[1]:
        return None, f"Test {index}: use 'Test name :: command :: needs=program'."
    needs = []
    if len(parts) == 3 and parts[2]:
        if not parts[2].startswith("needs="):
            return None, f"Test {index}: third section must begin with 'needs='."
        needs = [item.strip() for item in parts[2][6:].split(",") if item.strip()]
    return {"name": parts[0], "command": parts[1], "needs": needs}, None

def safe_repository_label(url):
    try:
        parsed = urllib.parse.urlsplit(url)
        if parsed.scheme and parsed.netloc:
            return urllib.parse.urlunsplit((parsed.scheme, parsed.netloc, parsed.path, "", ""))
    except Exception:
        pass
    return url.split("?", 1)[0]

def add_issue(stage, detail):
    issues.append({"stage": str(stage), "detail": clip_output(detail)})

# Validate basic configuration before launching any shell commands.
repo_url = str(repo_url).strip()
branch_or_ref = str(branch_or_ref).strip()
repo_label = safe_repository_label(repo_url)

if not repo_url or not (repo_url.startswith("https://") or repo_url.startswith("http://") or repo_url.startswith("git@")):
    add_issue("Configuration", "Repository URL must be https://, http://, or git@...")
if clone_depth < 0:
    add_issue("Configuration", "Clone depth must be 0 (full history) or a positive integer.")
if test_timeout_seconds < 1 or clone_timeout_seconds < 1:
    add_issue("Configuration", "Timeout values must be positive integers.")

test_specs = []
for position, raw_spec in enumerate([test_1, test_2, test_3, test_4, test_5], 1):
    parsed, error = parse_test(raw_spec, position)
    if error:
        add_issue("Configuration", error)
    elif parsed:
        test_specs.append(parsed)

if not test_specs:
    add_issue("Configuration", "No valid test commands were provided.")

if issues:
    print("Configuration issue(s) found. Correct the form fields and run this cell again.")
else:
    print("Project Test Console started.")
    print("Repository:", repo_label)
    print("Ref:", branch_or_ref or "(default branch)")
    print("Tests configured:", len(test_specs))

    # Read a Colab Secret only for Git clone. Public repositories can be cloned without it.
    try:
        if secret_name.strip():
            secret = userdata.get(secret_name.strip())
    except Exception:
        secret = None

    if shutil.which("git") is None:
        add_issue("Git", "git executable is not installed in this Colab runtime.")
    else:
        if WORKDIR.exists():
            shutil.rmtree(WORKDIR, ignore_errors=True)

        clone_env = os.environ.copy()
        clone_env["GIT_TERMINAL_PROMPT"] = "0"
        clone_env["GCM_INTERACTIVE"] = "Never"
        askpass_dir = None

        if secret:
            askpass_dir = pathlib.Path(tempfile.mkdtemp(prefix="project_test_askpass_"))
            askpass_file = askpass_dir / "askpass.sh"
            askpass_file.write_text(
                '#!/bin/sh\n'
                'case "$1" in\n'
                '  *Username*) printf "%s\\n" "$PROJECT_GIT_USERNAME" ;;\n'
                '  *) printf "%s\\n" "$PROJECT_GIT_TOKEN" ;;\n'
                'esac\n',
                encoding="utf-8"
            )
            askpass_file.chmod(0o700)
            clone_env.update({
                "GIT_ASKPASS": str(askpass_file),
                "PROJECT_GIT_USERNAME": git_username.strip() or "x-access-token",
                "PROJECT_GIT_TOKEN": secret
            })

        clone_command = ["git", "clone"]
        if clone_depth > 0:
            clone_command += ["--depth", str(clone_depth)]
        if branch_or_ref:
            clone_command += ["--single-branch", "--branch", branch_or_ref]
        clone_command += [repo_url, str(WORKDIR)]

        print("\n[1/3] Cloning repository…")
        try:
            cloned = subprocess.run(
                clone_command,
                env=clone_env,
                capture_output=True,
                text=True,
                timeout=clone_timeout_seconds
            )
            clone_output = redact((cloned.stdout or "") + (cloned.stderr or ""), secret)
            if cloned.returncode == 0:
                clone_ok = True
                commit = subprocess.check_output(
                    ["git", "rev-parse", "HEAD"],
                    cwd=str(WORKDIR),
                    text=True
                ).strip()
                print("Repository ready ✓ | commit:", commit[:12])
            else:
                add_issue("Git clone", f"exit={cloned.returncode}\n{clone_output or '(no output)'}")
        except subprocess.TimeoutExpired:
            add_issue("Git clone", f"Clone timed out after {clone_timeout_seconds} seconds.")
        except Exception as exc:
            add_issue("Git clone", f"{type(exc).__name__}: {exc}")
        finally:
            if askpass_dir:
                shutil.rmtree(askpass_dir, ignore_errors=True)
            clone_env.pop("PROJECT_GIT_TOKEN", None)
            clone_env.pop("PROJECT_GIT_USERNAME", None)
            secret = None

    if clone_ok:
        run_env = os.environ.copy()

        # User-defined setup commands run in order and do not inherit the GitHub token.
        for pos, command in enumerate([setup_command_1, setup_command_2, setup_command_3], 1):
            command = str(command).strip()
            if not command:
                continue
            print(f"\n[SETUP {pos}] {command}")
            code, duration, output = run_shell(command, WORKDIR, run_env, test_timeout_seconds)
            record = {
                "name": f"Setup {pos}",
                "command": command,
                "exit": code,
                "duration": duration,
                "output": clip_output(redact(output, secret))
            }
            setup_results.append(record)
            if code != 0:
                add_issue(record["name"], record["output"] or f"exit={code}")
                if not continue_after_failure:
                    break

        setup_blocked = any(item["exit"] != 0 for item in setup_results) and not continue_after_failure

        # Clang installation is optional and performed before the Clang test.
        clang_needed = any("clang++" in spec["needs"] for spec in test_specs)
        if not setup_blocked and install_clang_if_missing and clang_needed and shutil.which("clang++") is None:
            print("\n[TOOLS] Installing Clang…")
            code, duration, output = run_shell(
                "apt-get update -qq && apt-get install -y clang",
                WORKDIR,
                run_env,
                max(test_timeout_seconds, 600)
            )
            setup_results.append({
                "name": "Install Clang",
                "command": "apt-get update -qq && apt-get install -y clang",
                "exit": code,
                "duration": duration,
                "output": clip_output(output)
            })
            if code != 0:
                add_issue("Install Clang", output or f"exit={code}")

        print("\n[2/3] Running configured tests…")
        stop_tests = setup_blocked
        for spec in test_specs:
            if stop_tests:
                test_results.append({
                    "name": spec["name"], "status": "SKIP", "exit": None,
                    "duration": 0, "command": spec["command"],
                    "output": "Skipped because an earlier setup step failed and continue-after-failure is disabled."
                })
                continue

            missing = [name for name in spec["needs"] if shutil.which(name) is None]
            if missing:
                test_results.append({
                    "name": spec["name"], "status": "SKIP", "exit": None,
                    "duration": 0, "command": spec["command"],
                    "output": "Required executable(s) not found: " + ", ".join(missing)
                })
                print("SKIP:", spec["name"], "| missing:", ", ".join(missing))
                continue

            print("RUN :", spec["name"])
            code, duration, output = run_shell(
                spec["command"], WORKDIR, run_env, test_timeout_seconds
            )
            status = "PASS" if code == 0 else "FAIL"
            record = {
                "name": spec["name"],
                "status": status,
                "exit": code,
                "duration": duration,
                "command": spec["command"],
                "output": clip_output(redact(output, secret))
            }
            test_results.append(record)
            print(f"{status}: {spec['name']} ({duration}s)")
            if code != 0:
                add_issue(spec["name"], record["output"] or f"exit={code}")
                if not continue_after_failure:
                    stop_tests = True

# The report is plain text in the Colab cell output, ready to copy into ChatGPT.
elapsed = round(time.time() - started_all, 2)
passed = sum(item["status"] == "PASS" for item in test_results)
failed = sum(item["status"] == "FAIL" for item in test_results)
skipped = sum(item["status"] == "SKIP" for item in test_results)
setup_failed = any(item["exit"] != 0 for item in setup_results)

if issues or failed or setup_failed or not clone_ok:
    overall = "FAIL"
elif skipped:
    overall = "PARTIAL"
elif passed:
    overall = "PASS"
else:
    overall = "FAIL"

commit_url = ""
github_match = re.search(r"github\.com[:/]([^/]+/[^/.]+?)(?:\.git)?$", repo_url)
if github_match and commit != "NOT_AVAILABLE":
    commit_url = f"https://github.com/{github_match.group(1)}/commit/{commit}"

lines = [
    "PROJECT VALIDATION REPORT",
    f"Overall: {overall}",
    f"Repository: {repo_label}",
    f"Branch: {branch_or_ref or '(default branch)'}",
    f"Commit: {commit}",
    f"Commit URL: {commit_url or 'not available'}",
    f"Timestamp UTC: {datetime.datetime.now(datetime.timezone.utc).isoformat()}",
    f"Environment: {platform.platform()} | Python {platform.python_version()}",
    f"Duration: {elapsed}s",
    f"Results: {passed} PASS / {failed} FAIL / {skipped} SKIP",
    "",
    "SETUP RESULTS:"
]
if not setup_results:
    lines.append("(none)")
for item in setup_results:
    lines += [
        f"[{'PASS' if item['exit'] == 0 else 'FAIL'}] {item['name']} | exit={item['exit']} | {item['duration']}s",
        f"COMMAND: {item['command']}",
        item["output"],
        ""
    ]

lines.append("TEST RESULTS:")
if not test_results:
    lines.append("(none)")
for item in test_results:
    lines += [
        f"[{item['status']}] {item['name']} | exit={item['exit']} | {item['duration']}s",
        f"COMMAND: {item['command']}",
        item["output"],
        ""
    ]

lines.append("ISSUE POOL:")
if not issues:
    lines.append("(empty)")
for index, issue in enumerate(issues, 1):
    lines += [
        f"--- ISSUE {index}: {issue['stage']} ---",
        issue["detail"],
        "--- END ISSUE ---",
        ""
    ]

lines += ["", "END OF REPORT"]
report_text = "\n".join(lines)
REPORT_PATH.write_text(report_text, encoding="utf-8")

if clone_ok and not keep_workspace:
    shutil.rmtree(WORKDIR, ignore_errors=True)

print("\n" + "=" * 88)
print(report_text)
print("=" * 88)
print("\nREPORT SAVED:", REPORT_PATH)
print("Copy the report directly from this Colab cell output and paste it into ChatGPT.")